# 24 — Jev εναντίον Microsoft Decision-1: development μόνο

Προσθέτει το **`typesafe/jev-1.13-20260917`** στην ίδια ερώτηση πέντε θεμάτων
και στις ίδιες **100 dev κριτικές** του notebook 23. Χρησιμοποιεί την ήδη
ελεγμένη καταγραφή του Microsoft, χωρίς νέα αιτήματα σε εκείνο.

**Νέα συνεδρία Colab CPU.** Δεν χρειάζεται GPU, μοντέλα ή νέα CSV βαθμολόγησης.
Χρειάζεται το **decision_topics_inputs_20261012.zip** με τις κλειδωμένες ετικέτες.

Για πραγματική εκτέλεση, βάλε `ALLOW_JEV=True`, όρισε το δικό σου `MAX_COST_USD`
και ενεργοποίησε το Colab Secret **OPENROUTER_API_KEY** με capped key.
Από προεπιλογή κάνει μόνο ελέγχους, χωρίς κλειδί ή χρέωση. Έως 100 αιτήματα,
χωρίς retries. Το όριο βασίζεται στο κόστος που αναφέρει ο πάροχος· ένα αίτημα
μπορεί να το υπερβεί. Είναι software guard, όχι όριο λογαριασμού.

[Πρωτόκολλο επέκτασης](https://github.com/krimits/hotel-review-nlp/blob/8d40e875e2aa3e8dcaec2cdd1dc84260cbd910bb/docs/experiments/decision_topic_benchmark/JEV_EXTENSION.md).
Το Jev επιλέχθηκε αφού είδαμε Microsoft dev: η σύγκριση είναι επιπλέον
**development evidence**, όχι ανεξάρτητη επιβεβαίωση νικητή.
Δεν τρέχει test, δεν αναβαθμίζει το Space και δεν αλλάζει τις ανθρώπινες ετικέτες.

## 1. Ρυθμίσεις

Τρέξε τα κελιά με τη σειρά. Διάλεξε μόνος σου budget πριν ενεργοποιήσεις το API.
Το `0.10`, για παράδειγμα, θα ήταν δικό σου επιλεγμένο software limit σε USD·
δεν έχει επιλεγεί εδώ ως εγκεκριμένη χρέωση. Κράτησε το κλειδί στα Colab Secrets.

In [ ]:
from pathlib import Path
CAPTURE_IS_CURRENT = False
RUN_EXIT_CODE = None
ALLOW_JEV = False
MAX_COST_USD = None
SOURCE_COMMIT = "8d40e875e2aa3e8dcaec2cdd1dc84260cbd910bb"
BASE = Path("/content/hotel-jev-primary-topics-dev-v1")
REPO = BASE / "repo"
ENV = BASE / "env"
PYTHON = ENV / "bin/python"
INPUTS = BASE / "inputs"
OUT = REPO / "runs/jev_primary_topics/colab_dev"
REPO_URL = "https://github.com/krimits/hotel-review-nlp.git"


## 2. Καρφιτσωμένος κώδικας και απομονωμένο CPU περιβάλλον

Εγκαθιστά μόνο τα πακέτα των αριθμητικών ελέγχων σε δικό του venv.
Ελέγχει πραγματικές εκδόσεις και τη θέση των imports, χωρίς αλλαγές στα global
Gradio/diffusers του Colab. Η εκτέλεση μοντέλου γίνεται αποκλειστικά μέσω API.

In [ ]:
import json
import math
import os
import subprocess
import venv

BASE.mkdir(parents=True, exist_ok=True)
clean_env = {k: v for k, v in os.environ.items()
             if not k.startswith(("PYTHON", "PIP"))
             and k not in {"OPENROUTER_API_KEY", "TYPESAFE_API_KEY", "HF_TOKEN"}}
logs = []
def run(command, cwd=None):
    result = subprocess.run([str(x) for x in command], cwd=cwd, env=clean_env,
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    logs.append(result.stdout)
    (BASE / "setup_execution.txt").write_text("\n".join(logs))
    if result.returncode:
        print(result.stdout[-6000:])
        raise RuntimeError(f"Command failed, exit {result.returncode}; see setup_execution.txt")
    return result.stdout

if not REPO.exists():
    run(["git", "clone", "--no-checkout", REPO_URL, REPO])
    run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO)
assert run(["git", "rev-parse", "HEAD"], cwd=REPO).strip() == SOURCE_COMMIT
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=True, system_site_packages=False).create(ENV)
run([PYTHON, "-I", "-m", "pip", "--isolated", "install", "--no-user", "-r",
     REPO / "configs/decision_topic_sample_requirements.txt"])
check = """
import importlib, importlib.metadata, json, pathlib, sys
expected = {"numpy": "2.5.4", "scipy": "1.18.1", "scikit-learn": "1.9.1"}
assert pathlib.Path(sys.prefix).resolve() == pathlib.Path(sys.argv[1]).resolve()
assert sys.prefix != sys.base_prefix
report = {"executable": sys.executable, "packages": {}}
for name, version in expected.items():
    module = importlib.import_module("sklearn" if name == "scikit-learn" else name)
    assert importlib.metadata.version(name) == version == module.__version__
    assert pathlib.Path(module.__file__).resolve().is_relative_to(pathlib.Path(sys.prefix).resolve())
    report["packages"][name] = {"version": version, "origin": module.__file__}
print(json.dumps(report, indent=2))
"""
runtime_json = run([PYTHON, "-I", "-c", check, ENV])
(BASE / "runtime_preflight.json").write_text(runtime_json)
print(runtime_json)
print("Source:", SOURCE_COMMIT)


## 3. Ανέβασμα του ήδη ελεγμένου ZIP

Επίλεξε **decision_topics_inputs_20261012.zip**, όχι το αρχικό ZIP των κενών
φύλλων και όχι το αποτέλεσμα του notebook 23. Το script ελέγχει την allowlist,
τα hashes και τις ανθρώπινες ετικέτες πριν από οποιοδήποτε API αίτημα.
Μετά ελέγχει και ξαναϋπολογίζει την αποθηκευμένη σύγκριση του Microsoft.
Η επανάληψη δεν αντικαθιστά υπάρχον φάκελο ή προηγούμενη εκτέλεση.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one locked input ZIP")
_, content = uploaded.popitem()
package = BASE / "decision_topics_inputs.zip"
package.write_bytes(content)
del content
print(run([PYTHON, "-I", REPO / "scripts/unpack_decision_topic_inputs.py",
           "--package", package, "--output", INPUTS], cwd=REPO))
validation = run([PYTHON, "-I", REPO / "scripts/benchmark_jev_primary_topics.py", "check",
                  "--inputs", INPUTS], cwd=REPO)
(BASE / "inputs_check.json").write_text(validation)
print(validation)


## 4. Μόνο Jev, με ρητή ενεργοποίηση

Το secret διαβάζεται μόνο όταν `ALLOW_JEV=True` και έχεις ορίσει θετικό budget.
Ο κώδικας απορρίπτει αλλαγή του model ID ή κακοσχηματισμένες πιθανότητες.
Σε αποτυχία/άγνωστο κόστος σταματά τα επόμενα αιτήματα και κρατά όλες τις
προγραμματισμένες γραμμές. Exit 1 σημαίνει ελλιπή εκτέλεση: πάρε το ZIP ακόμη
και τότε. Δεν υπάρχει δεύτερη αυτόματη προσπάθεια ή σιωπηλή αλλαγή μοντέλου.

In [ ]:
run_env = dict(clean_env)
RUN_EXIT_CODE = None
CAPTURE_IS_CURRENT = False
if not ALLOW_JEV:
    print("API disabled: input/reference checks only; no secret lookup or provider request.")
else:
    if OUT.exists():
        raise ValueError("Existing capture: start a fresh session; no old report is a new run")
    if (isinstance(MAX_COST_USD, bool) or not isinstance(MAX_COST_USD, (int, float))
            or not math.isfinite(MAX_COST_USD) or MAX_COST_USD <= 0):
        raise ValueError("Choose a positive USD observed-spend budget before enabling Jev")
    from google.colab import userdata
    secret = userdata.get("OPENROUTER_API_KEY")
    if not secret:
        raise ValueError("Enable the OPENROUTER_API_KEY Colab Secret")
    run_env["OPENROUTER_API_KEY"] = secret
    command = [PYTHON, "-I", REPO / "scripts/benchmark_jev_primary_topics.py", "run",
               "--inputs", INPUTS, "--output", OUT,
               "--allow-external-api", "--max-cost-usd", str(MAX_COST_USD)]
    CAPTURE_IS_CURRENT = True
    try:
        with (BASE / "execution.txt").open("w") as log:
            execution = subprocess.run([str(x) for x in command], cwd=REPO,
                                       env=run_env, stdout=log, stderr=subprocess.STDOUT)
        RUN_EXIT_CODE = execution.returncode
    finally:
        run_env.pop("OPENROUTER_API_KEY", None)
        secret = None
    print("Exit code:", RUN_EXIT_CODE)
    if RUN_EXIT_CODE:
        print("Incomplete run: download the diagnostics ZIP. Do not rerun an inspected capture.")


## 5. Μετρικές και σύγκριση

Η κύρια μετρική είναι το raw macro-F1. Το ECE μετά τη βαθμονόμηση είναι
**in-sample dev diagnostic**. Στο confidence ≥0.7 δες μαζί πόσες κρατήθηκαν
και πόσα λάθη είχαν, όχι μόνο το ECE. Κανένα retained σημαίνει απροσδιόριστο
selective error, όχι μηδέν.

Η Microsoft μέτρηση χρόνου είναι ιστορική· η Jev από νέα συνεδρία. Οι χρόνοι
περιλαμβάνουν δίκτυο και δεν αποδεικνύουν εγγενή ταχύτητα αρχιτεκτονικής.
Το USD/1.000 είναι αναγωγή του αναφερόμενου κόστους αυτών των αιτημάτων.

In [ ]:
from IPython.display import Markdown, display
report_path = OUT / "run.json"
report = json.loads(report_path.read_text()) if CAPTURE_IS_CURRENT and report_path.exists() else None
if report is None:
    print("No Jev report: API was disabled or failed before capture. Download available diagnostics.")
else:
    def fmt(value):
        return "unknown / undefined" if value is None else f"{value:.4f}"
    lines = ["| Arm | Successful | Raw macro-F1 | Raw accuracy | Raw ECE | Dev-fitted ECE | Retained >=0.7 | Retained error | p95 ms | USD/1,000 attempts |",
             "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for arm, name in [("jev", "Jev"), ("decision", "Microsoft reference")]:
        s = report["summary"][arm]; raw = s["raw"] or {}; cal = s["calibrated"] or {}
        lines.append(f"| {name} | {s['successful']}/{s['planned']} | {fmt(raw.get('macro_f1'))} | {fmt(raw.get('accuracy'))} | {fmt(raw.get('ece'))} | {fmt(cal.get('ece'))} | {cal.get('retained', 'undefined')} | {fmt(cal.get('selective_error_rate'))} | {fmt(s.get('p95_request_ms'))} | {fmt(s.get('cost_usd_per_1000_attempts'))} |")
    display(Markdown("\n".join(lines)))
    print("Complete:", report["execution_complete"])
    print("Gold support:", (report["summary"]["decision"]["raw"] or {}).get("support"))
    print("Paired comparison:", report["paired_comparison"])
    print("No automatic winner, test prediction or Space update.")


In [ ]:
if report is not None and report["paired_comparison"] is not None:
    import matplotlib.pyplot as plt
    comparison = report["paired_comparison"]
    values = [report["summary"][arm]["raw"]["macro_f1"] for arm in ["jev", "decision"]]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.7), constrained_layout=True)
    bars = axes[0].bar(["Jev", "Microsoft"], values, color=["#1565c0", "#617080"])
    axes[0].set_ylim(0, 1)
    axes[0].set_ylabel("Macro-F1, five classes")
    axes[0].set_title("Same 100 dev reviews")
    axes[0].bar_label(bars, labels=[f"{v:.4f}" for v in values], padding=3)
    low, high = comparison["delta_macro_f1_percentile_ci_95"]
    delta = comparison["delta_macro_f1_jev_minus_microsoft"]
    axes[1].hlines(0, low, high, color="#1565c0", linewidth=3)
    axes[1].scatter([delta], [0], color="#1565c0", zorder=3)
    axes[1].axvline(0, color="#617080", linestyle="--")
    axes[1].set_yticks([])
    axes[1].set_xlabel("Macro-F1 difference: Jev minus Microsoft")
    axes[1].set_title("Paired bootstrap: 95% dev diagnostic interval")
    axes[1].text(0.5, 0.85, f"delta {delta:.4f}; [{low:.4f}, {high:.4f}]",
                 transform=axes[1].transAxes, ha="center")
    fig.savefig(BASE / "dev_comparison.png", dpi=160)
    plt.show()
    print("Rare classes and unverified reviewer independence limit inference. This is not a production score.")


## 6. Κατέβασμα αποτελέσματος ή διαγνωστικών

Το ZIP περιέχει μόνο model outputs, χρόνους/κόστος, hashes, επιλεγμένα logs
και το aggregate chart. Δεν περιέχει τις κριτικές, τα ανθρώπινα φύλλα ή κλειδιά.
Η `human_queue.csv` είναι κατάλογος model referrals· **δεν χρειάζεται να τη
συμπληρώσεις** για αυτή τη σύγκριση.

Σε σφάλμα setup/εισόδου, τρέξε χειροκίνητα αυτό το τελευταίο κελί για τα διαθέσιμα διαγνωστικά. Παλιό capture δεν εξάγεται ως νέα εκτέλεση.

In [ ]:
import hashlib
import zipfile
from google.colab import files
output_package = BASE / "jev_primary_topics_dev_comparison.zip"
allowed_results = ["jev_predictions.jsonl", "microsoft_reference_predictions.json",
                   "run.json", "human_queue.csv", "freeze.json"]
allowed_diagnostics = ["setup_execution.txt", "runtime_preflight.json", "inputs_check.json",
                       "execution.txt", "dev_comparison.png"]
with zipfile.ZipFile(output_package, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in allowed_results:
        path = OUT / name
        if globals().get("CAPTURE_IS_CURRENT", False) and path.exists():
            archive.write(path, "comparison/" + name)
    for name in allowed_diagnostics:
        path = BASE / name
        if path.exists() and (name != "dev_comparison.png" or globals().get("CAPTURE_IS_CURRENT", False)):
            archive.write(path, name)
print("ZIP SHA-256:", hashlib.sha256(output_package.read_bytes()).hexdigest())
print("API exit code:", globals().get("RUN_EXIT_CODE"))
files.download(str(output_package))


## Τι στέλνεις και τι ακολουθεί

Στείλε **jev_primary_topics_dev_comparison.zip**, ακόμη και με exit 1.
Δεν υπάρχει ακόμη μετρημένο αποτέλεσμα Jev σε αυτό το notebook template.
Θα ελέγξουμε τη διαφορά, τις κατηγορίες και τα λάθη των retained περιπτώσεων.
Η υπεροχή σε dev δεν προωθεί αυτόματα μοντέλο: ακολουθούν επιλογή παγωμένου
configuration και μία αξιολόγηση στο ανέγγιχτο test.

Η Qwen T4 precision διόρθωση παραμένει ξεχωριστό βήμα πριν την τελική πλήρη
σύγκριση. Η θεματική ταξινόμηση δεν αποδεικνύει ποιότητα εντοπισμού εκκρεμών
παραπόνων, ακριβών τεκμηρίων ή χρήσιμων μέτρων για τον ξενοδόχο.